# 2. Limpeza dos Dados

Esta etapa aplica os tratamentos indicados nas conclusões da inspeção (`01_data_inspection.ipynb`, seção 1.11) e gera a base que será usada na análise exploratória e na modelagem.

| Item | Situação encontrada na inspeção | Tratamento |
|---|---|---|
| `Unnamed: 32` | 100% vazia, gerada pela vírgula ao final do cabeçalho | Remover |
| `id` | Identificador único, sem valor preditivo | Remover |
| `concave points_*` | Espaço no nome da coluna | Renomear para `concave_points_*` |
| `diagnosis` | Categórica (`M`/`B`) | Converter para `1`/`0` |
| Valores ausentes | Nenhum nas variáveis de medida | Verificar e confirmar |
| Duplicatas | Nenhuma | Verificar e confirmar |
| Zeros em concavidade | 13 exames, valores plausíveis | Manter |
| Outliers (IQR) | 171 exames (30,1%), 2/3 deles malignos | Manter |

O arquivo bruto em `data/raw/` não é modificado. A base tratada é salva em `data/processed/breast_cancer_wisconsin_clean.csv`.


## 2.1 Importação das bibliotecas


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 12)


## 2.2 Carregamento da base


In [2]:
DATA_PATH = "../data/raw/breast_cancer_wisconsin.csv"
OUTPUT_PATH = "../data/processed/breast_cancer_wisconsin_clean.csv"

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")


Linhas: 569
Colunas: 33


A cópia `df` recebe todos os tratamentos; `df_raw` é preservada para a comparação final.


## 2.3 Remoção da coluna vazia `Unnamed: 32`


In [3]:
empty_cols = df.columns[df.isnull().all()].tolist()
print(f"Colunas totalmente vazias: {empty_cols}")

df = df.drop(columns=empty_cols)
print(f"Colunas após a remoção: {df.shape[1]}")


Colunas totalmente vazias: ['Unnamed: 32']
Colunas após a remoção: 32


A coluna é identificada de forma genérica (qualquer coluna 100% vazia) para que o notebook continue correto caso o CSV seja regenerado sem a vírgula final. Apenas `Unnamed: 32` atende ao critério.


## 2.4 Remoção da coluna `id`


In [4]:
print(f"IDs únicos: {df['id'].nunique()} de {len(df)} registros")

df = df.drop(columns=["id"])
print(f"Colunas após a remoção: {df.shape[1]}")


IDs únicos: 569 de 569 registros
Colunas após a remoção: 31


O `id` identifica cada exame de forma única e não carrega informação sobre o tumor. Mantê-lo como variável preditora permitiria ao modelo memorizar registros em vez de aprender padrões. Como não há IDs repetidos, a coluna pode ser descartada sem perda de informação.


## 2.5 Padronização dos nomes das colunas


In [5]:
renamed = {col: col.replace(" ", "_") for col in df.columns if " " in col}
df = df.rename(columns=renamed)

renamed


{'concave points_mean': 'concave_points_mean',
 'concave points_se': 'concave_points_se',
 'concave points_worst': 'concave_points_worst'}

Os espaços em `concave points_mean`, `concave points_se` e `concave points_worst` são substituídos por `_`. Isso evita a necessidade de aspas ou colchetes em consultas e mantém o padrão `caracteristica_sufixo` das demais colunas.


## 2.6 Conversão da variável alvo `diagnosis`


In [6]:
print(f"Valores encontrados: {sorted(df['diagnosis'].unique())}")

df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0}).astype("int64")

pd.DataFrame({
    "quantidade": df["diagnosis"].value_counts(),
    "percentual": (df["diagnosis"].value_counts(normalize=True) * 100).round(2),
})


Valores encontrados: ['B', 'M']


,quantidade,percentual
diagnosis,,
0,357,62.74
1,212,37.26


A variável alvo passa a ser numérica: `1` para maligno e `0` para benigno. O maligno é a classe positiva porque é o caso que o modelo precisa detectar; assim, métricas como recall e F1-score ficam alinhadas ao erro mais grave do problema (falso negativo). A contagem se mantém em 212 malignos e 357 benignos, confirmando que nenhum valor ficou sem mapeamento.


## 2.7 Valores ausentes e registros duplicados


In [7]:
print(f"Valores ausentes: {df.isnull().sum().sum()}")
print(f"Registros duplicados: {df.duplicated().sum()}")

rows_before = len(df)
df = df.drop_duplicates()
print(f"Registros removidos por duplicidade: {rows_before - len(df)}")


Valores ausentes: 0
Registros duplicados: 0
Registros removidos por duplicidade: 0


Como antecipado na inspeção, não há valores ausentes nas variáveis de medida nem registros duplicados. A chamada a `drop_duplicates` é mantida por segurança e não remove nenhuma linha. Nenhuma estratégia de imputação é necessária.


## 2.8 Zeros e outliers: decisão de manter


In [8]:
concavity_cols = [col for col in df.columns if col.startswith(("concavity", "concave_points"))]
zero_rows = (df[concavity_cols] == 0).any(axis=1)
print(f"Exames com zero nas colunas de concavidade: {zero_rows.sum()}")

features = df.drop(columns=["diagnosis"])
q1, q3 = features.quantile(0.25), features.quantile(0.75)
iqr = q3 - q1
is_outlier = (features < q1 - 1.5 * iqr) | (features > q3 + 1.5 * iqr)
rows_with_outlier = is_outlier.any(axis=1)

print(f"Exames com ao menos um outlier (IQR): {rows_with_outlier.sum()} ({rows_with_outlier.mean() * 100:.1f}%)")
print("Outliers por classe (1 = maligno):")
print(df.loc[rows_with_outlier, "diagnosis"].value_counts().to_string())


Exames com zero nas colunas de concavidade: 13
Exames com ao menos um outlier (IQR): 171 (30.1%)
Outliers por classe (1 = maligno):
diagnosis
1    114
0     57


Os zeros nas colunas de concavidade representam núcleos sem reentrâncias no contorno, um valor fisicamente possível, e por isso não são tratados como ausentes.

Os outliers concentram-se na classe maligna (114 dos 171 exames). Removê-los descartaria justamente os casos mais extremos que o modelo precisa aprender a reconhecer. A decisão é mantê-los e tratar as diferenças de escala e assimetria no pré-processamento, com padronização, transformação logarítmica ou escalonador robusto, conforme o modelo.


## 2.9 Validação da base limpa


In [9]:
assert df.shape == (569, 31), df.shape
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0
assert set(df["diagnosis"].unique()) == {0, 1}
assert df.drop(columns=["diagnosis"]).dtypes.eq("float64").all()
assert not any(" " in col for col in df.columns)

print("Validação concluída sem erros.")
df.info()


Validação concluída sem erros.
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    int64  
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave_points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_s

In [10]:
df.head()


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,...,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,...,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,...,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,...,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,...,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,...,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


A base limpa tem 569 registros e 31 colunas: `diagnosis` (inteiro, `0`/`1`) e 30 variáveis de medida (`float64`). Não há valores ausentes, duplicatas ou nomes de coluna com espaço.


## 2.10 Salvamento da base tratada


In [11]:
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

df_saved = pd.read_csv(OUTPUT_PATH)
pd.testing.assert_frame_equal(df_saved, df)
print(f"Base salva em {OUTPUT_PATH}: {df_saved.shape[0]} linhas x {df_saved.shape[1]} colunas")


Base salva em ../data/processed/breast_cancer_wisconsin_clean.csv: 569 linhas x 31 colunas


O arquivo é gravado sem o índice do `DataFrame` e relido em seguida para garantir que o conteúdo salvo é idêntico ao que está em memória.


## 2.11 Conclusões da limpeza


In [12]:
pd.DataFrame({
    "bruta": [df_raw.shape[0], df_raw.shape[1], df_raw.isnull().sum().sum(), str(df_raw["diagnosis"].dtype)],
    "limpa": [df.shape[0], df.shape[1], df.isnull().sum().sum(), str(df["diagnosis"].dtype)],
}, index=["linhas", "colunas", "valores ausentes", "tipo de diagnosis"])


,bruta,limpa
linhas,569,569
colunas,33,31
valores ausentes,569,0
tipo de diagnosis,str,int64


| Tratamento | Resultado |
|---|---|
| Remoção de `Unnamed: 32` | 1 coluna removida |
| Remoção de `id` | 1 coluna removida |
| Renomeação de `concave points_*` | 3 colunas renomeadas |
| Conversão de `diagnosis` | `M` → `1`, `B` → `0` |
| Valores ausentes | Nenhum encontrado, nenhuma imputação |
| Duplicatas | Nenhuma encontrada, nenhuma linha removida |
| Zeros e outliers | Mantidos |

A base passou de 33 para 31 colunas e manteve os 569 exames. Nenhum valor foi alterado ou imputado; as únicas mudanças são estruturais (remoção de colunas, renomeação e codificação do alvo). A base tratada em `data/processed/breast_cancer_wisconsin_clean.csv` é o ponto de partida da análise exploratória (`03_eda.ipynb`) e da modelagem.

**Pendências para as etapas seguintes**

- Padronização das variáveis, dada a diferença de escala (área na casa dos milhares, suavidade abaixo de 1).
- Avaliar transformação logarítmica ou escalonador robusto para as variáveis `_se`, muito assimétricas.
- Considerar seleção de atributos por causa da multicolinearidade entre raio, perímetro e área e entre as versões `_mean` e `_worst`.
- Usar separação estratificada entre treino e teste por causa do desbalanceamento moderado das classes.
